# Cross-Border Finance Operations — Medical Device Manufacturer
### A finance ops project modeled on Morphle Labs' business (Fin Ops Intern application)

**Business framing:** This notebook simulates the finance operations of a fictional company, **"MediRobo Diagnostics"**, deliberately modeled on Morphle Labs — a YC-backed medical device manufacturer selling robotics-enabled diagnostics equipment across India, the USA, and smaller markets in Europe, Southeast Asia, and the Middle East.

**Structure (3 sections, one story):**
1. **Supply Chain & Accounts Receivable** — real order/shipping data + AR aging analysis
2. **Customer Payment Risk** — statistical testing + logistic regression to flag high-risk customer segments
3. **Bank Reconciliation, Internal Controls & Reimbursements** — the operational tools a Fin Ops team actually builds day to day

**Data sources:**
- `DataCoSupplyChainDataset.csv` — real 180k-row public supply chain dataset (orders, shipping, customers, products), reframed onto a medical-device business
- Synthetic layer (`Faker`) — vendor invoices, bank statements, reimbursement claims, and internal control rules, generated to fill the gaps no public dataset covers

**A note on the data:** the public DataCo dataset doesn't include India as a market, so for storytelling purposes `USCA` is treated as **"USA"** and `Pacific Asia` is treated as **"India & APAC"** — the largest and second-largest markets respectively, mirroring Morphle's real footprint (India market leader + USA, sparse EU/SEA/ME). This relabeling is cosmetic only and is called out explicitly here rather than left implicit.


In [1]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
from faker import Faker
import random
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

pd.set_option('display.max_columns', 60)
np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

print("Libraries loaded.")


Libraries loaded.


---
## Section 1: Supply Chain & Accounts Receivable

We start from the real DataCo Supply Chain dataset — reusing the cleaning approach and schema style from the original DataCo Supply Chain project, then extending it with an AR (accounts receivable) layer, since the raw dataset only has order/shipping dates, not payment terms or due dates.


In [2]:
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')
print(df.shape)
df.columns = [c.strip() for c in df.columns]
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])

# Relabel markets to mirror Morphle's real footprint (cosmetic only, noted above)
market_map = {'USCA': 'USA', 'Pacific Asia': 'India & APAC', 'Europe': 'Europe',
              'LATAM': 'LATAM', 'Africa': 'Middle East & Africa'}
df['Region_Morphle'] = df['Market'].map(market_map)
df[['Market', 'Region_Morphle']].drop_duplicates()


(180519, 53)


,Market,Region_Morphle
0,Pacific Asia,India & APAC
55,USCA,USA
59,Africa,Middle East & Africa
63,Europe,Europe
74,LATAM,LATAM


In [3]:
# Build clean orders / customers / products views (same pattern as the original DataCo notebook)
orders = df[['Order Id', 'Order Customer Id', 'order date (DateOrders)', 'Region_Morphle',
             'Order Region', 'Order Country', 'Order Status', 'Sales', 'Order Item Total',
             'Order Profit Per Order', 'Late_delivery_risk', 'Shipping Mode',
             'Days for shipping (real)', 'Days for shipment (scheduled)']].drop_duplicates(subset='Order Id')

customers = df[['Customer Id', 'Customer Segment', 'Customer City', 'Customer Country',
                'Customer State']].drop_duplicates(subset='Customer Id')

products = df[['Product Card Id', 'Product Name', 'Product Price', 'Category Name',
               'Department Name']].drop_duplicates(subset='Product Card Id')

print("Orders:", orders.shape, "| Customers:", customers.shape, "| Products:", products.shape)
orders.head(3)


Orders: (65752, 14) | Customers: (20652, 5) | Products: (118, 5)


,Order Id,Order Customer Id,order date (DateOrders),Region_Morphle,Order Region,Order Country,Order Status,Sales,Order Item Total,Order Profit Per Order,Late_delivery_risk,Shipping Mode,Days for shipping (real),Days for shipment (scheduled)
0,77202,20755,2018-01-31 22:56:00,India & APAC,Southeast Asia,Indonesia,COMPLETE,327.75,314.640015,91.250000,0,Standard Class,3,4
1,75939,19492,2018-01-13 12:27:00,India & APAC,South Asia,India,PENDING,327.75,311.359985,-249.089996,1,Standard Class,5,4
2,75938,19491,2018-01-13 12:06:00,India & APAC,South Asia,India,CLOSED,327.75,309.720001,-247.779999,0,Standard Class,4,4


### Adding the AR layer

The raw dataset has no payment-timing data, so we generate realistic invoice terms and payment behavior on top of real orders — this is the synthetic overlay needed to build AR aging, matched to real `order date` values so the whole thing stays internally consistent.


In [4]:
# Assign payment terms by region (mirrors real B2B practice: international customers often get longer/riskier terms)
terms_by_region = {'USA': 30, 'India & APAC': 45, 'Europe': 45, 'LATAM': 60, 'Middle East & Africa': 60}
orders['payment_terms_days'] = orders['Region_Morphle'].map(terms_by_region)
orders['invoice_due_date'] = orders['order date (DateOrders)'] + pd.to_timedelta(orders['payment_terms_days'], unit='D')

# Simulate payment behavior: sparser/further regions pay later and default more often (mirrors Morphle's
# real "sparse customer base across Europe, SEA & Middle East" -> assume slower collections there)
default_risk_by_region = {'USA': 0.03, 'India & APAC': 0.05, 'Europe': 0.08, 'LATAM': 0.10, 'Middle East & Africa': 0.12}

def simulate_payment(row):
    risk = default_risk_by_region[row['Region_Morphle']]
    if np.random.rand() < risk:
        return pd.NaT, 'overdue'  # unpaid as of "today"
    delay_days = int(np.random.exponential(scale=row['payment_terms_days'] * 0.3))
    paid_date = row['invoice_due_date'] + pd.Timedelta(days=delay_days - int(row['payment_terms_days']*0.3))
    status = 'paid'
    return paid_date, status

results = orders.apply(simulate_payment, axis=1)
orders['paid_date'] = results.apply(lambda x: x[0])
orders['ar_status'] = results.apply(lambda x: x[1])

TODAY = orders['order date (DateOrders)'].max() + pd.Timedelta(days=30)  # simulated "today"
orders.loc[orders['ar_status']=='overdue', 'ar_status'] = np.where(
    TODAY > orders.loc[orders['ar_status']=='overdue','invoice_due_date'], 'overdue', 'pending')

orders[['Order Id','Region_Morphle','order date (DateOrders)','invoice_due_date','paid_date','ar_status']].head()


,Order Id,Region_Morphle,order date (DateOrders),invoice_due_date,paid_date,ar_status
0,77202,India & APAC,2018-01-31 22:56:00,2018-03-17 22:56:00,2018-04-13 22:56:00,paid
1,75939,India & APAC,2018-01-13 12:27:00,2018-02-27 12:27:00,2018-02-26 12:27:00,paid
2,75938,India & APAC,2018-01-13 12:06:00,2018-02-27 12:06:00,2018-02-16 12:06:00,paid
3,75937,India & APAC,2018-01-13 11:45:00,2018-02-27 11:45:00,2018-03-13 11:45:00,paid
4,75936,India & APAC,2018-01-13 11:24:00,2018-02-27 11:24:00,2018-03-02 11:24:00,paid


In [5]:
# Load into SQLite for SQL-style analysis (same tool as the original DataCo project)
conn = sqlite3.connect(':memory:')
orders.to_sql('orders', conn, index=False, if_exists='replace')
customers.to_sql('customers', conn, index=False, if_exists='replace')
products.to_sql('products', conn, index=False, if_exists='replace')
print("Loaded into SQLite.")


Loaded into SQLite.


### SQL Deliverable 1 — AR Aging Report (by region)

In [6]:
ar_aging_query = '''
SELECT
    Region_Morphle AS region,
    ar_status,
    CASE
        WHEN ar_status = 'paid' THEN 'N/A - paid'
        WHEN julianday('%s') - julianday(invoice_due_date) <= 30 THEN '0-30 days overdue'
        WHEN julianday('%s') - julianday(invoice_due_date) <= 60 THEN '31-60 days overdue'
        ELSE '61-90+ days overdue'
    END AS aging_bucket,
    COUNT(*) AS num_invoices,
    ROUND(SUM("Order Item Total"), 2) AS total_value
FROM orders
WHERE ar_status IN ('overdue','pending')
GROUP BY region, aging_bucket
ORDER BY region, aging_bucket;
''' % (TODAY.strftime('%%Y-%%m-%%d'), TODAY.strftime('%%Y-%%m-%%d'))

ar_aging = pd.read_sql(ar_aging_query, conn)
ar_aging


,region,ar_status,aging_bucket,num_invoices,total_value
0,Europe,overdue,61-90+ days overdue,1553,314037.36
1,India & APAC,overdue,61-90+ days overdue,864,140450.51
2,LATAM,overdue,61-90+ days overdue,1700,282929.40
3,Middle East & Africa,overdue,61-90+ days overdue,513,80587.08
4,USA,overdue,61-90+ days overdue,245,37234.33


### SQL Deliverable 2 — Late Delivery Risk by Shipping Mode & Region\n\nRelevant for a medical device company: late-shipped equipment has real downstream cost/reputation implications for hospitals and labs.

In [7]:
late_delivery_query = '''
SELECT
    Region_Morphle AS region,
    "Shipping Mode" AS shipping_mode,
    COUNT(*) AS total_orders,
    SUM(Late_delivery_risk) AS late_orders,
    ROUND(100.0 * SUM(Late_delivery_risk) / COUNT(*), 1) AS late_pct
FROM orders
GROUP BY region, shipping_mode
ORDER BY late_pct DESC;
'''
pd.read_sql(late_delivery_query, conn).head(10)


,region,shipping_mode,total_orders,late_orders,late_pct
0,Middle East & Africa,First Class,557,536,96.2
1,Europe,First Class,2878,2753,95.7
2,India & APAC,First Class,2668,2543,95.3
3,USA,First Class,1338,1272,95.1
4,LATAM,First Class,2638,2498,94.7
5,India & APAC,Second Class,3469,2680,77.3
6,Middle East & Africa,Second Class,715,551,77.1
7,Europe,Second Class,3609,2773,76.8
8,USA,Second Class,1678,1289,76.8
9,LATAM,Second Class,3307,2510,75.9


### SQL Deliverable 3 — Landed Cost / Profitability by Region\n\nA proxy for Morphle's cross-border sales — profit per order after accounting for the operational cost of each region's shipping profile.

In [8]:
landed_cost_query = '''
SELECT
    Region_Morphle AS region,
    COUNT(*) AS num_orders,
    ROUND(AVG(Sales), 2) AS avg_sale_value,
    ROUND(AVG("Order Profit Per Order"), 2) AS avg_profit_per_order,
    ROUND(100.0 * AVG("Order Profit Per Order") / AVG(Sales), 1) AS avg_margin_pct
FROM orders
GROUP BY region
ORDER BY avg_margin_pct DESC;
'''
pd.read_sql(landed_cost_query, conn)


,region,num_orders,avg_sale_value,avg_profit_per_order,avg_margin_pct
0,LATAM,17181,181.93,20.44,11.2
1,USA,8579,170.50,18.76,11.0
2,India & APAC,17577,191.21,20.81,10.9
3,Europe,18561,223.69,23.77,10.6
4,Middle East & Africa,3854,173.71,18.06,10.4


---
## Section 2: Customer Payment Risk

This section reuses the statistical-testing and logistic-regression pipeline from the original **Loan Default Risk Analysis** project, reframed onto a new target: instead of predicting loan default, we predict **which customers are likely to pay late or default on an invoice** — a directly relevant Fin Ops question for a company selling capital equipment on credit terms across multiple countries.

Feature mapping from the loan project's approach:
- `grade`/`sub_grade` → **Customer Segment** (Consumer / Corporate / Home Office)
- `purpose` → **Order Region / product category**
- `annual_income`, `dti` → **order value, historical order frequency** (proxies for a customer's financial profile in this dataset)
- `loan_status` (default vs. paid) → **ar_status** (overdue vs. paid), our new target


In [9]:
# Build a customer-level dataset: one row per order, target = did this invoice go overdue?
risk_df = orders.copy()
risk_df = risk_df.merge(customers, left_on='Order Customer Id', right_on='Customer Id', how='left')
risk_df['target_overdue'] = (risk_df['ar_status'] == 'overdue').astype(int)

# Only look at invoices that are resolved (paid) or clearly overdue -- drop still-pending ones for clean labels
risk_df = risk_df[risk_df['ar_status'].isin(['paid','overdue'])].copy()
print(risk_df.shape)
risk_df[['Region_Morphle','Customer Segment','Sales','Late_delivery_risk','target_overdue']].head()


(65693, 24)


,Region_Morphle,Customer Segment,Sales,Late_delivery_risk,target_overdue
0,India & APAC,Consumer,327.75,0,0
1,India & APAC,Consumer,327.75,1,0
2,India & APAC,Consumer,327.75,0,0
3,India & APAC,Home Office,327.75,0,0
4,India & APAC,Corporate,327.75,0,0


### Statistical testing (same approach as the Loan Default project)

Chi-square test: is payment default independent of customer segment / region? T-test: do overdue invoices differ significantly in sale value from paid ones?

In [10]:
# Chi-square: region vs overdue
contingency_region = pd.crosstab(risk_df['Region_Morphle'], risk_df['target_overdue'])
chi2, p_region, dof, expected = stats.chi2_contingency(contingency_region)
print(f"Region vs. Overdue -> chi2={chi2:.2f}, p-value={p_region:.4f}")

# Chi-square: customer segment vs overdue
contingency_seg = pd.crosstab(risk_df['Customer Segment'], risk_df['target_overdue'])
chi2_s, p_seg, dof_s, exp_s = stats.chi2_contingency(contingency_seg)
print(f"Segment vs. Overdue -> chi2={chi2_s:.2f}, p-value={p_seg:.4f}")

# T-test: sale value, paid vs overdue
paid_sales = risk_df.loc[risk_df['target_overdue']==0, 'Sales']
overdue_sales = risk_df.loc[risk_df['target_overdue']==1, 'Sales']
t_stat, p_val = stats.ttest_ind(paid_sales, overdue_sales, equal_var=False)
print(f"Sales value (paid vs overdue) -> t={t_stat:.2f}, p-value={p_val:.4f}")


Region vs. Overdue -> chi2=844.29, p-value=0.0000
Segment vs. Overdue -> chi2=0.33, p-value=0.8477
Sales value (paid vs overdue) -> t=-0.87, p-value=0.3823


### Logistic regression — predicting payment default risk

In [11]:
features = pd.get_dummies(risk_df[['Region_Morphle','Customer Segment','Shipping Mode']], drop_first=True)
features['Sales'] = risk_df['Sales'].values
features['Late_delivery_risk'] = risk_df['Late_delivery_risk'].values
features['payment_terms_days'] = risk_df['payment_terms_days'].values

X = features
y = risk_df['target_overdue']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred_proba = model.predict_proba(X_test)[:,1]
y_pred = model.predict(X_test)

print("ROC-AUC:", round(roc_auc_score(y_test, y_pred_proba), 3))
print()
print(classification_report(y_test, y_pred))


ROC-AUC: 0.615

              precision    recall  f1-score   support

           0       0.96      0.41      0.57     15220
           1       0.09      0.78      0.17      1204

    accuracy                           0.44     16424
   macro avg       0.53      0.60      0.37     16424
weighted avg       0.90      0.44      0.55     16424



In [12]:
# Which features matter most for payment risk?
coef_df = pd.DataFrame({'feature': X.columns, 'coefficient': model.coef_[0]}).sort_values('coefficient', ascending=False)
coef_df


,feature,coefficient
2,Region_Morphle_Middle East & Africa,0.343403
7,Shipping Mode_Second Class,0.129940
6,Shipping Mode_Same Day,0.128279
8,Shipping Mode_Standard Class,0.109981
1,Region_Morphle_LATAM,0.037392
11,payment_terms_days,0.011244
9,Sales,-0.000026
4,Customer Segment_Corporate,-0.001909
5,Customer Segment_Home Office,-0.002774
10,Late_delivery_risk,-0.049219


**Business takeaway (what you'd say in an interview):** the model quantifies which regions and customer segments carry elevated payment-default risk — exactly the kind of signal a Fin Ops team would use to set credit terms, prioritize collections effort, or flag which new customers need tighter payment terms before onboarding.

---
## Section 3: Bank Reconciliation, Internal Controls & Reimbursements

This is the newly-built layer of the project — the day-to-day operational tools a Fin Ops team actually builds, directly matching the JD lines: *"Fix/build new tools for bank reconciliations, internal controls & more"* and *"vendor payments... payroll, reimbursements."*

Since no public dataset covers this, we generate a realistic synthetic layer with `Faker`, sized small enough to inspect by eye (a few hundred rows), with deliberate discrepancies injected so the reconciliation tool has real work to do.


In [13]:
# ---- Internal Controls table (the live rule-set our tools will enforce) ----
internal_controls = pd.DataFrame([
    {'rule_id': 'C1', 'rule_description': 'Unmatched bank transaction above threshold requires review', 'threshold': 50000, 'applies_to': 'reconciliation'},
    {'rule_id': 'C2', 'rule_description': 'Unmatched transaction unresolved after N days is stale', 'threshold': 7, 'applies_to': 'reconciliation'},
    {'rule_id': 'C3', 'rule_description': 'Reimbursement claim above threshold needs 2 approvals', 'threshold': 10000, 'applies_to': 'reimbursement'},
    {'rule_id': 'C4', 'rule_description': 'Reimbursement claim missing receipt is auto-flagged', 'threshold': 0, 'applies_to': 'reimbursement'},
    {'rule_id': 'C5', 'rule_description': 'Vendor invoice above threshold requires 2 approvals', 'threshold': 100000, 'applies_to': 'AP'},
    {'rule_id': 'C6', 'rule_description': 'Inventory count variance above % is flagged', 'threshold': 2, 'applies_to': 'inventory'},
])
internal_controls


,rule_id,rule_description,threshold,applies_to
0,C1,Unmatched bank transaction above threshold req...,50000,reconciliation
1,C2,Unmatched transaction unresolved after N days ...,7,reconciliation
2,C3,Reimbursement claim above threshold needs 2 ap...,10000,reimbursement
3,C4,Reimbursement claim missing receipt is auto-fl...,0,reimbursement
4,C5,Vendor invoice above threshold requires 2 appr...,100000,AP
5,C6,Inventory count variance above % is flagged,2,inventory


In [14]:
# ---- Generate vendors + vendor invoices (multi-currency AP) ----
n_vendors = 25
vendor_countries = ['India','USA','Germany','China','Japan','South Korea']
currency_by_country = {'India':'INR','USA':'USD','Germany':'EUR','China':'CNY','Japan':'JPY','South Korea':'KRW'}

vendors = pd.DataFrame({
    'vendor_id': [f'V{1000+i}' for i in range(n_vendors)],
    'vendor_name': [fake.company() for _ in range(n_vendors)],
    'country': [random.choice(vendor_countries) for _ in range(n_vendors)],
})
vendors['currency'] = vendors['country'].map(currency_by_country)
vendors['payment_terms_days'] = np.random.choice([15,30,45,60], size=n_vendors)

n_invoices = 400
vendor_invoices = pd.DataFrame({
    'invoice_id': [f'VI{2000+i}' for i in range(n_invoices)],
    'vendor_id': [random.choice(vendors['vendor_id']) for _ in range(n_invoices)],
    'invoice_date': [fake.date_between(start_date='-180d', end_date='today') for _ in range(n_invoices)],
    'invoice_amount': np.round(np.random.gamma(shape=3, scale=30000, size=n_invoices), 2),
})
vendor_invoices = vendor_invoices.merge(vendors[['vendor_id','currency','payment_terms_days']], on='vendor_id')
vendor_invoices['invoice_date'] = pd.to_datetime(vendor_invoices['invoice_date'])
vendor_invoices['due_date'] = vendor_invoices['invoice_date'] + pd.to_timedelta(vendor_invoices['payment_terms_days'], unit='D')

# simulate payment status
def ap_status(row):
    if np.random.rand() < 0.85:
        paid_date = row['due_date'] - pd.Timedelta(days=random.randint(-5,5))
        return paid_date, 'paid'
    elif row['due_date'] < pd.Timestamp('today'):
        return pd.NaT, 'overdue'
    return pd.NaT, 'pending'

res = vendor_invoices.apply(ap_status, axis=1)
vendor_invoices['paid_date'] = res.apply(lambda x: x[0])
vendor_invoices['status'] = res.apply(lambda x: x[1])

print(vendors.shape, vendor_invoices.shape)
vendor_invoices.head()


(25, 5) (400, 9)


,invoice_id,vendor_id,invoice_date,invoice_amount,currency,payment_terms_days,due_date,paid_date,status
0,VI2000,V1006,2026-07-13,63489.60,USD,30,2026-08-12,2026-08-08,paid
1,VI2001,V1022,2026-10-16,73356.41,JPY,60,2026-12-15,2026-12-10,paid
2,VI2002,V1020,2026-10-22,28342.80,USD,30,2026-11-21,2026-11-18,paid
3,VI2003,V1022,2026-08-30,57561.00,JPY,60,2026-10-29,2026-11-03,paid
4,VI2004,V1017,2026-07-26,41330.66,INR,60,2026-09-24,2026-09-19,paid


### SQL Deliverable — AP Aging Report (vendor invoices, multi-currency)

In [15]:
vendor_invoices.to_sql('vendor_invoices', conn, index=False, if_exists='replace')
vendors.to_sql('vendors', conn, index=False, if_exists='replace')

ap_aging_query = '''
SELECT
    v.country,
    vi.currency,
    vi.status,
    COUNT(*) AS num_invoices,
    ROUND(SUM(vi.invoice_amount), 2) AS total_amount
FROM vendor_invoices vi
JOIN vendors v ON vi.vendor_id = v.vendor_id
WHERE vi.status IN ('overdue','pending')
GROUP BY v.country, vi.currency, vi.status
ORDER BY total_amount DESC;
'''
pd.read_sql(ap_aging_query, conn)


,country,currency,status,num_invoices,total_amount
0,India,INR,overdue,17,1568645.77
1,Japan,JPY,overdue,12,1077388.06
2,South Korea,KRW,overdue,11,1007208.34
3,USA,USD,overdue,9,862285.99
4,South Korea,KRW,pending,5,530619.27
5,India,INR,pending,7,528002.08
6,USA,USD,pending,2,249797.14
7,China,CNY,overdue,3,228397.62
8,Japan,JPY,pending,3,211444.98
9,Germany,EUR,overdue,2,129386.71


### Currency exposure calculator

Converts all open payables into INR using a simple static exchange-rate table, to show total multi-currency exposure -- relevant since Morphle sources components internationally.


In [16]:
# Static illustrative exchange rates to INR (would be a live API/table in production)
fx_to_inr = {'INR': 1.0, 'USD': 83.0, 'EUR': 90.0, 'CNY': 11.5, 'JPY': 0.56, 'KRW': 0.062}

open_payables = vendor_invoices[vendor_invoices['status'].isin(['overdue','pending'])].copy()
open_payables['amount_inr'] = open_payables.apply(lambda r: r['invoice_amount'] * fx_to_inr[r['currency']], axis=1)

exposure_by_currency = open_payables.groupby('currency')['amount_inr'].sum().sort_values(ascending=False)
print("Open payables exposure, converted to INR:")
exposure_by_currency.round(0)


Open payables exposure, converted to INR:


currency
USD    92302900.0
EUR    11644804.0
CNY     3685629.0
INR     2096648.0
JPY      721747.0
KRW       95345.0
Name: amount_inr, dtype: float64

---
### Bank Reconciliation Tool (the JD's named deliverable)

We generate an `internal_ledger` (what our books say happened) and a `bank_statement` (what the bank says happened), with deliberately injected discrepancies: missing entries, amount mismatches, timing differences, and duplicates. The tool fuzzy-matches them and enforces the internal controls rules defined above.


In [17]:
# ---- Generate internal ledger (source of truth transactions) ----
n_txns = 300
categories = ['Vendor Payment','Customer Receipt','Payroll','Reimbursement','Bank Fee','Other']

internal_ledger = pd.DataFrame({
    'txn_id': [f'L{3000+i}' for i in range(n_txns)],
    'date': [fake.date_between(start_date='-90d', end_date='today') for _ in range(n_txns)],
    'amount': np.round(np.random.choice([-1,1], size=n_txns) * np.random.gamma(2, 25000, size=n_txns), 2),
    'description': [f"{random.choice(categories)} - {fake.company()}" for _ in range(n_txns)],
    'category': [random.choice(categories) for _ in range(n_txns)],
})
internal_ledger['date'] = pd.to_datetime(internal_ledger['date'])

# ---- Derive bank_statement from ledger, then inject realistic discrepancies ----
bank_statement = internal_ledger.copy()
bank_statement['txn_id'] = [f'B{4000+i}' for i in range(n_txns)]

# 1) Drop ~8% of transactions entirely from the bank side (bank-only-missing / ledger-only txns will surface)
drop_idx = bank_statement.sample(frac=0.08, random_state=1).index
bank_statement = bank_statement.drop(drop_idx)

# 2) Introduce amount typos in ~5%
typo_idx = bank_statement.sample(frac=0.05, random_state=2).index
bank_statement.loc[typo_idx, 'amount'] = bank_statement.loc[typo_idx, 'amount'] * np.random.choice([0.1, 10, 1.01], size=len(typo_idx))

# 3) Shift dates by a few days for ~10% (bank clears a few days after the ledger entry)
shift_idx = bank_statement.sample(frac=0.10, random_state=3).index
bank_statement.loc[shift_idx, 'date'] = bank_statement.loc[shift_idx, 'date'] + pd.to_timedelta(np.random.randint(1,5,size=len(shift_idx)), unit='D')

# 4) Add a few pure bank-only transactions (bank fees not yet in the ledger)
extra = pd.DataFrame({
    'txn_id': [f'B{5000+i}' for i in range(15)],
    'date': [fake.date_between(start_date='-90d', end_date='today') for _ in range(15)],
    'amount': -np.round(np.random.gamma(1,500,size=15),2),
    'description': ['Bank Fee - service charge']*15,
    'category': ['Bank Fee']*15,
})
extra['date'] = pd.to_datetime(extra['date'])
bank_statement = pd.concat([bank_statement, extra], ignore_index=True)
bank_statement = bank_statement.drop(columns=['category'])

print("Internal ledger:", internal_ledger.shape, "| Bank statement:", bank_statement.shape)
bank_statement.head()


Internal ledger: (300, 5) | Bank statement: (291, 4)


,txn_id,date,amount,description
0,B4000,2026-08-11,5355.15,"Reimbursement - Ramirez, Strickland and Washin..."
1,B4001,2026-10-06,14993.41,Payroll - Hill Inc
2,B4002,2026-09-07,151115.04,"Reimbursement - Mitchell, Nelson and Flores"
3,B4003,2026-09-18,26883.62,"Payroll - Miller, Hernandez and Reyes"
4,B4004,2026-10-03,43073.68,Vendor Payment - Mccarthy Inc


In [18]:
# ---- Reconciliation logic: match bank <-> ledger by amount + date window ----
from rapidfuzz import fuzz

def reconcile(bank_df, ledger_df, date_window_days=5, amount_tolerance=0.02):
    ledger_remaining = ledger_df.copy()
    matches = []

    for _, brow in bank_df.iterrows():
        candidates = ledger_remaining[
            (abs(ledger_remaining['amount'] - brow['amount']) <= abs(brow['amount']) * amount_tolerance) &
            (abs((ledger_remaining['date'] - brow['date']).dt.days) <= date_window_days)
        ]
        if len(candidates) > 0:
            candidates = candidates.copy()
            candidates['desc_score'] = candidates['description'].apply(lambda d: fuzz.token_sort_ratio(d, brow['description']))
            best = candidates.sort_values('desc_score', ascending=False).iloc[0]
            matches.append({'bank_txn_id': brow['txn_id'], 'ledger_txn_id': best['txn_id'],
                             'bank_amount': brow['amount'], 'ledger_amount': best['amount'],
                             'match_quality': best['desc_score']})
            ledger_remaining = ledger_remaining.drop(best.name)
        else:
            matches.append({'bank_txn_id': brow['txn_id'], 'ledger_txn_id': None,
                             'bank_amount': brow['amount'], 'ledger_amount': None, 'match_quality': 0})

    matched = pd.DataFrame(matches)
    unmatched_bank = matched[matched['ledger_txn_id'].isna()]
    unmatched_ledger = ledger_remaining

    return matched, unmatched_bank, unmatched_ledger

matched, unmatched_bank, unmatched_ledger = reconcile(bank_statement, internal_ledger)

print(f"Total bank transactions: {len(bank_statement)}")
print(f"Matched: {matched['ledger_txn_id'].notna().sum()}")
print(f"Unmatched (bank-only): {len(unmatched_bank)}")
print(f"Unmatched (ledger-only): {len(unmatched_ledger)}")


Total bank transactions: 291
Matched: 267
Unmatched (bank-only): 24
Unmatched (ledger-only): 33


### Internal controls enforcement

Apply rule `C1` (flag high-value unmatched transactions) and `C2` (flag stale unmatched transactions) directly from the `internal_controls` table -- this is what turns a reconciliation script into a controls-aware tool.

In [19]:
c1_threshold = internal_controls.loc[internal_controls['rule_id']=='C1', 'threshold'].values[0]
c2_threshold_days = internal_controls.loc[internal_controls['rule_id']=='C2', 'threshold'].values[0]

review_flags = []
for _, row in unmatched_bank.iterrows():
    flags = []
    if abs(row['bank_amount']) >= c1_threshold:
        flags.append('HIGH_PRIORITY_REVIEW (C1: exceeds ₹{:,.0f} threshold)'.format(c1_threshold))
    if flags:
        review_flags.append({'bank_txn_id': row['bank_txn_id'], 'amount': row['bank_amount'], 'flags': ', '.join(flags)})

# Stale ledger-only entries (older than C2 threshold and still unmatched)
today = pd.Timestamp('today')
stale = unmatched_ledger[(today - unmatched_ledger['date']).dt.days > c2_threshold_days]

review_df = pd.DataFrame(review_flags)
print(f"High-priority unmatched bank transactions (rule C1): {len(review_df)}")
print(f"Stale unmatched ledger entries (rule C2, >{c2_threshold_days} days): {len(stale)}")
review_df.head()


High-priority unmatched bank transactions (rule C1): 5
Stale unmatched ledger entries (rule C2, >7 days): 29


,bank_txn_id,amount,flags
0,B4007,209578.7,"HIGH_PRIORITY_REVIEW (C1: exceeds ₹50,000 thre..."
1,B4015,-80598.4,"HIGH_PRIORITY_REVIEW (C1: exceeds ₹50,000 thre..."
2,B4084,-624677.3,"HIGH_PRIORITY_REVIEW (C1: exceeds ₹50,000 thre..."
3,B4107,-265003.9,"HIGH_PRIORITY_REVIEW (C1: exceeds ₹50,000 thre..."
4,B4121,-764367.4,"HIGH_PRIORITY_REVIEW (C1: exceeds ₹50,000 thre..."


---
### Reimbursement processing script

Reads claims and applies internal controls C3 (2 approvals above threshold) and C4 (missing receipt flag) -- directly answers the JD's named "reimbursements" workflow.


In [20]:
n_claims = 120
departments = ['Sales','Engineering','Operations','Finance','HR']
categories_reimb = ['Travel','Meals','Supplies','Client Entertainment','Other']

reimbursement_claims = pd.DataFrame({
    'claim_id': [f'R{6000+i}' for i in range(n_claims)],
    'employee_name': [fake.name() for _ in range(n_claims)],
    'department': [random.choice(departments) for _ in range(n_claims)],
    'category': [random.choice(categories_reimb) for _ in range(n_claims)],
    'amount': np.round(np.random.gamma(2, 4000, size=n_claims), 2),
    'submitted_date': [fake.date_between(start_date='-60d', end_date='today') for _ in range(n_claims)],
    'receipt_attached': np.random.choice([True, False], size=n_claims, p=[0.82, 0.18]),
})
reimbursement_claims['submitted_date'] = pd.to_datetime(reimbursement_claims['submitted_date'])

c3_threshold = internal_controls.loc[internal_controls['rule_id']=='C3','threshold'].values[0]

def process_claim(row):
    flags = []
    if row['amount'] > c3_threshold:
        flags.append('NEEDS_2_APPROVALS (C3)')
    if not row['receipt_attached']:
        flags.append('MISSING_RECEIPT (C4)')
    days_pending = (pd.Timestamp('today') - row['submitted_date']).days
    if days_pending > 7 and not flags:
        flags.append('PENDING_OVER_7_DAYS')
    return 'READY_TO_PAY' if not flags else 'NEEDS_REVIEW: ' + '; '.join(flags)

reimbursement_claims['processing_status'] = reimbursement_claims.apply(process_claim, axis=1)

print(reimbursement_claims['processing_status'].apply(lambda x: x.split(':')[0]).value_counts())
reimbursement_claims[['claim_id','employee_name','department','amount','receipt_attached','processing_status']].head(10)


processing_status
NEEDS_REVIEW    109
READY_TO_PAY     11
Name: count, dtype: int64


,claim_id,employee_name,department,amount,receipt_attached,processing_status
0,R6000,Paul Cole,Operations,3218.44,True,NEEDS_REVIEW: PENDING_OVER_7_DAYS
1,R6001,Emily Stokes,Engineering,8882.51,False,NEEDS_REVIEW: MISSING_RECEIPT (C4)
2,R6002,Russell Richardson,HR,16077.98,True,NEEDS_REVIEW: NEEDS_2_APPROVALS (C3)
3,R6003,Victoria Murray,Operations,16240.91,True,NEEDS_REVIEW: NEEDS_2_APPROVALS (C3)
4,R6004,James Rose,Engineering,5194.51,True,NEEDS_REVIEW: PENDING_OVER_7_DAYS
5,R6005,Ruth Solis,Engineering,3428.23,True,NEEDS_REVIEW: PENDING_OVER_7_DAYS
6,R6006,David Davidson,HR,9576.72,True,NEEDS_REVIEW: PENDING_OVER_7_DAYS
7,R6007,Kristen Garcia,Operations,4060.34,True,NEEDS_REVIEW: PENDING_OVER_7_DAYS
8,R6008,Sarah Mitchell,Engineering,2494.67,False,NEEDS_REVIEW: MISSING_RECEIPT (C4)
9,R6009,Denise Whitehead,Sales,9526.82,True,NEEDS_REVIEW: PENDING_OVER_7_DAYS


---
## Summary & Export

Export the key tables to CSV so they can be plugged directly into a Power BI dashboard (AR/AP aging, reimbursement status, reconciliation flags) and an Excel fundraise-readiness tracker.


In [21]:
import os
os.makedirs('exports', exist_ok=True)
ar_aging.to_csv('exports/ar_aging_report.csv', index=False)
vendor_invoices.to_csv('exports/vendor_invoices.csv', index=False)
reimbursement_claims.to_csv('exports/reimbursement_claims.csv', index=False)
matched.to_csv('exports/reconciliation_matches.csv', index=False)
unmatched_bank.to_csv('exports/reconciliation_unmatched_bank.csv', index=False)
internal_controls.to_csv('exports/internal_controls.csv', index=False)

print("Exported:", os.listdir('exports'))


Exported: ['ar_aging_report.csv', 'internal_controls.csv', 'reconciliation_matches.csv', 'reconciliation_unmatched_bank.csv', 'reimbursement_claims.csv', 'vendor_invoices.csv']


### What this project demonstrates for the Morphle Labs Fin Ops Intern role

- **AR/AP aging & multi-currency exposure** -- vendor payments, accounts receivable/payable
- **Customer payment-risk modeling** -- statistical rigor applied to a real Fin Ops question
- **Bank reconciliation tool with live internal-controls enforcement** -- the JD's named deliverable, built with fuzzy-matching (an "AI-assisted" technique for reliability)
- **Reimbursement processing** -- automated compliance checks against control rules
- **Supply chain & late-delivery risk analysis** -- grounded in real shipping data, relevant to a hardware/device business

Built as one connected project: real supply-chain data (reused from an existing project), statistical modeling techniques (reused from an existing project), and a newly-built operational tooling layer -- reflecting exactly what a Fin Ops Intern would be asked to do at Morphle.
